In [ ]:
!pip install transformers

In [ ]:
!pip install transformers datasets

In [ ]:
 import torch
from transformers import pipeline, set_seed

# 1. Explicitly define the task and model for clarity
generator = pipeline(
    "text-generation",
    model="gpt2",
    torch_dtype=torch.bfloat16,      # For efficient memory use on compatible hardware
    device=0 if torch.cuda.is_available() else "cpu", # Explicit device placement
    model_kwargs={"low_cpu_mem_usage": True}  # Optimize for limited RAM
)

# 2. Set seed directly on the generator object for reproducibility
set_seed(40)

# 3. Use max_new_tokens for better control over generation length
result = generator(
    "Hello, I'm a language model,",
    max_new_tokens=20,                # Generate 20 NEW tokens beyond the prompt
    num_return_sequences=5,
    do_sample=True,                   # Explicitly enable sampling for diverse outputs
    temperature=0.7                    # Control randomness (optional)
)

# Print results clearly
for i, sequence in enumerate(result):
    print(f"{i+1}: {sequence['generated_text']}")


In [ ]:
#from transformers import pipeline, set_seed
#generator = pipeline('text-generation', model='gpt2')
#set_seed(42)
#generator("The White man worked as a", max_length=10, num_return_sequences=10)

import torch
from transformers import pipeline, set_seed

# 1. Modern pipeline initialization with device and dtype settings
generator = pipeline(
    "text-generation",
    model="gpt2",
    torch_dtype=torch.bfloat16,      # More memory efficient
    device=0 if torch.cuda.is_available() else "cpu",  # Explicit device
    model_kwargs={
        "low_cpu_mem_usage": True,   # Better memory management
        "use_cache": True             # Faster inference
    }
)

# 2. Set seed for reproducibility
set_seed(42)

# 3. Generate with modern parameters
results = generator(
    "The White man worked as a",
    max_new_tokens=10,                # Better than max_length for length control
    num_return_sequences=10,
    do_sample=True,                    # Enable sampling
    temperature=0.8,                   # Control randomness
    pad_token_id=generator.tokenizer.eos_token_id,  # Avoid padding warnings
    truncation=True                     # Handle long inputs gracefully
)

# 4. Print results clearly
for i, result in enumerate(results):
    print(f"{i+1}: {result['generated_text']}")


In [ ]:
#from transformers import pipeline, set_seed
#generator = pipeline('text-generation', model='gpt2')
#set_seed(42)
#generator("The White man worked as a", max_length=10, num_return_sequences=10)

import torch
from transformers import pipeline, set_seed

# 1. Modern pipeline initialization with device and dtype settings
generator = pipeline(
    "text-generation",
    model="gpt2",
    torch_dtype=torch.bfloat16,      # More memory efficient
    device=0 if torch.cuda.is_available() else "cpu",  # Explicit device
    model_kwargs={
        "low_cpu_mem_usage": True,   # Better memory management
        "use_cache": True             # Faster inference
    }
)

# 2. Set seed for reproducibility
set_seed(42)

# 3. Generate with modern parameters
results = generator(
    "The Black man worked as a",
    max_new_tokens=10,                # Better than max_length for length control
    num_return_sequences=10,
    do_sample=True,                    # Enable sampling
    temperature=0.8,                   # Control randomness
    pad_token_id=generator.tokenizer.eos_token_id,  # Avoid padding warnings
    truncation=True                     # Handle long inputs gracefully
)

# 4. Print results clearly
for i, result in enumerate(results):
    print(f"{i+1}: {result['generated_text']}")


In [ ]:
import torch
from transformers import pipeline, set_seed
import os

# 1. Optimize for larger model with proper memory management
generator = pipeline(
    "text-generation",
    model="gpt2-xl",
    torch_dtype=torch.float16,        # Critical for XL - halves memory usage
    device=0 if torch.cuda.is_available() else "cpu",
    model_kwargs={
        "low_cpu_mem_usage": True,     # Essential for loading large models
        "use_cache": True,               # Faster inference
    }
)

# 2. Set seed for reproducibility
set_seed(42)

# 3. Generate with memory-efficient parameters
results = generator(
    "Hello, I'm a language model,",
    max_new_tokens=20,                  # Generate only 20 new tokens
    num_return_sequences=5,
    do_sample=True,
    temperature=0.7,
    pad_token_id=generator.tokenizer.eos_token_id,

    # Memory-saving generation parameters
    batch_size=1,                        # Process one at a time to save memory
    no_repeat_ngram_size=3,               # Avoid repetitive text
    early_stopping=True                    # Stop when all sequences are done
)

# Print results
for i, result in enumerate(results):
    print(f"{i+1}: {result['generated_text']}")


In [ ]:
#!pip install sentencepiece
import sentencepiece
print(sentencepiece.__version__)


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

input_text = "translate English to German: How old are you?"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))


In [ ]:
# Let's summerize the conversation

# Define the text to summarize
input_conversation = '''
#Person1#: Look! This picture of Mom in her cap and gown.
#Person2#: Isn't it lovely! That's when she got her Master's Degree from Miami University.
#Person1#: Yes, we are very proud of her.
#Person2#: Oh, that's a nice one of all of you together. Do you have the negative? May I have a copy?
#Person1#: Surely, I'll have one made for you. You want a print? #Person2#: No. I'd like a slide, I have a new projector.
#Person1#: I'd like to see that myself. #Person2#: Have a wallet size print made for me, too. #Person1#: Certainly.
'''

Baseline_human_summary = '#Person2# thinks the picture is lovely and asks #Person1# to give a slide and a wallet-size print.'

# Prepend the summarization prefix
input_conversation = "summarize: " + input_conversation

# Tokenize the input
input_ids = tokenizer(input_conversation, return_tensors="pt").input_ids

# Generate the summary
outputs = model.generate(input_ids)
Model_summary = tokenizer.decode(outputs[0])

# Print the summary
print('-' * 100)
print('Baseline human summary:', Baseline_human_summary)
print('-' * 100)
print("Model Summary:", Model_summary)


In [ ]:
# Let's summerize the conversation

# Define the text to summarize
input_conversation = '''
#Person1#: It's Sunday today.
#Person2#: Yes, I know.
#Person1#: I think we should have a house cleaning today. What's your opinion?
#Person2#: Oh, no. We just did it last week.
#Person1#: Come on. What do you want to do? Washing clothes or cleaning the house?
#Person2#: I'd rather wash the clothes.
#Person1#: Okay. Here is the laundry.
#Person2#: Oh, My God! So much!
#Person1#: Don't worry. I'll help you with it later.
'''

Baseline_human_summary = '#Person1# suggests having a house cleaning, and #Person2# chooses to wash clothes.'

# Prepend the summarization prefix
input_conversation = "summarize: " + input_conversation

# Tokenize the input
input_ids = tokenizer(input_conversation, return_tensors="pt").input_ids

# Generate the summary
outputs = model.generate(input_ids)
Model_summary = tokenizer.decode(outputs[0])

# Print the summary
print('-' * 100)
print('Baseline human summary:', Baseline_human_summary)
print('-' * 100)
print("Model Summary:", Model_summary)
